In [54]:
import os
import json
import requests
import pandas as pd
from datetime import datetime, timedelta
from nvd_client import NvdApi
from cwe2.database import Database
from mitreattack.stix20 import MitreAttackData
from neo4j import GraphDatabase

In [56]:
nvd_api = NvdApi()
cves_by_date = nvd_api.get_cve_by_date(
    per_page=100,
    offset=0,
    publish_start_date=datetime.now() - timedelta(days=2),
    publish_end_date=datetime.now()
)
cves_by_date

{'resultsPerPage': 22,
 'startIndex': 0,
 'totalResults': 22,
 'format': 'NVD_CVE',
 'version': '2.0',
 'timestamp': '2025-10-20T17:23:09.726',
 'vulnerabilities': [{'cve': {'id': 'CVE-2025-47410',
    'sourceIdentifier': 'security@apache.org',
    'published': '2025-10-18T16:15:35.557',
    'lastModified': '2025-10-20T14:15:40.713',
    'vulnStatus': 'Received',
    'cveTags': [],
    'descriptions': [{'lang': 'en',
      'value': 'Apache Geode is vulnerable to CSRF attacks through GET requests to the Management and Monitoring REST API that could allow an attacker who has tricked a user into giving up their Geode session credentials to submit malicious commands on the target system on behalf of the authenticated user.\n\n\nThis issue affects Apache Geode: versions 1.10 through 1.15.1\n\nUsers are recommended to upgrade to version 1.15.2, which fixes the issue.'}],
    'metrics': {'cvssMetricV31': [{'source': '134c704f-9b21-4f2e-91b3-4a467353bcc0',
       'type': 'Secondary',
       'c

In [43]:
db = Database()
cwe_data = db.get_top_25_cwe()[0]
cwe_data

Weakness(cwe_id='20', name='Improper Input Validation', weakness_abstraction='Class', status='Stable', description='The product receives input or data, but it does not validate or incorrectly validates that the input has the properties that are required to process the data safely and correctly.', extended_description="Input validation is a frequently-used technique for checking potentially dangerous inputs in order to ensure that the inputs are safe for processing within the code, or when communicating with other components. When software does not validate input properly, an attacker is able to craft the input in a form that is not expected by the rest of the application. This will lead to parts of the system receiving unintended input, which may result in altered control flow, arbitrary control of a resource, or arbitrary code execution. Input validation is not the only technique for processing input, however. Other techniques attempt to transform potentially-dangerous input into some

In [46]:
cwe_data.detection_methods, cwe_data.potential_mitigations, cwe_data.related_weaknesses, cwe_data.observed_examples

('::METHOD:Automated Static Analysis:DESCRIPTION:Some instances of improper input validation can be detected using automated static analysis. A static analysis tool might allow the user to specify which application-specific methods or functions perform input validation; the tool might also have built-in knowledge of validation frameworks such as Struts. The tool may then suppress or de-prioritize any associated warnings. This allows the analyst to focus on areas of the software in which input validation does not appear to be present. Except in the cases described in the previous paragraph, automated static analysis might not be able to recognize when proper input validation is being performed, leading to false positives - i.e., warnings that do not have any security consequences or require any code changes.::METHOD:Manual Static Analysis:DESCRIPTION:When custom input validation is required, such as when enforcing business rules, manual analysis is necessary to ensure that the validatio

In [4]:
os.environ['NEO4J_URI'] = 'neo4j://127.0.0.1:7687'
os.environ['NEO4J_USERNAME'] = 'neo4j'
os.environ['NEO4J_PASSWORD'] = 'test1234'

In [6]:
driver = GraphDatabase.driver(
    os.environ['NEO4J_URI'],
    auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD'])
)


def test_connection(driver):
    with driver.session() as session:
        result = session.run("RETURN 'Hello World' as message")
        print(result.single()["message"])


test_connection(driver)
driver.close()

Hello World


# CWE Data Integration with Neo4j

In [9]:
def populate_cwe_data(driver):
    """Populate CWE (Common Weakness Enumeration) data"""
    # Download CWE XML from https://cwe.mitre.org/data/xml/cwec_latest.xml
    # Parse and populate - implementation depends on XML structure
    
    sample_cwes = [
        {"id": "CWE-79", "name": "Cross-site Scripting", "description": "Improper Neutralization of Input"},
        {"id": "CWE-89", "name": "SQL Injection", "description": "Improper Neutralization of Special Elements"},
        {"id": "CWE-20", "name": "Improper Input Validation", "description": "Product does not validate input properly"}
    ]
    
    with driver.session() as session:
        for cwe in sample_cwes:
            session.run("""
                MERGE (c:CWE {id: $cwe_id})
                SET c.name = $name,
                    c.description = $description
            """, cwe_id=cwe['id'], name=cwe['name'], description=cwe['description'])

# CAPEC Data Integration with Neo4j

In [10]:
def populate_capec_data(driver):
    """Populate CAPEC (Common Attack Pattern Enumeration) data"""
    sample_capecs = [
        {"id": "CAPEC-66", "name": "SQL Injection", "description": "Adversary injects SQL commands"},
        {"id": "CAPEC-85", "name": "AJAX Fingerprinting", "description": "Adversary explores web application"}
    ]
    
    with driver.session() as session:
        for capec in sample_capecs:
            session.run("""
                MERGE (c:CAPEC {id: $capec_id})
                SET c.name = $name,
                    c.description = $description
            """, capec_id=capec['id'], name=capec['name'], description=capec['description'])

# Relationship Mapping

In [11]:
def create_schema_constraints(driver):
    """Create database constraints and indexes"""
    with driver.session() as session:
        # Unique constraints
        session.run("CREATE CONSTRAINT cve_id IF NOT EXISTS FOR (c:CVE) REQUIRE c.id IS UNIQUE")
        session.run("CREATE CONSTRAINT cwe_id IF NOT EXISTS FOR (c:CWE) REQUIRE c.id IS UNIQUE")
        session.run("CREATE CONSTRAINT capec_id IF NOT EXISTS FOR (c:CAPEC) REQUIRE c.id IS UNIQUE")
        
        # Indexes for performance
        session.run("CREATE INDEX cve_severity IF NOT EXISTS FOR (c:CVE) ON (c.severity)")
        session.run("CREATE INDEX cve_date IF NOT EXISTS FOR (c:CVE) ON (c.published_date)")

def create_relationships(driver):
    """Create relationships between entities"""
    with driver.session() as session:
        # CVE to CWE relationships
        session.run("""
            MATCH (cve:CVE), (cwe:CWE)
            WHERE cve.description CONTAINS 'injection' AND cwe.id = 'CWE-89'
            MERGE (cve)-[:EXPLOITS]->(cwe)
        """)

        # CWE to CAPEC relationships
        session.run("""
            MATCH (cwe:CWE), (capec:CAPEC)
            WHERE cwe.id = 'CWE-89' AND capec.id = 'CAPEC-66'
            MERGE (cwe)-[:ENABLES]->(capec)
        """)

# GraphRAG Implementation with langchain

In [12]:
from langchain.vectorstores import Neo4jVector
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import GraphCypherQAChain
from langchain.graphs import Neo4jGraph

c:\Users\ishani.kathuria\projects\TrustworthyRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
class CybersecurityGraphRAG:
    def __init__(self, neo4j_url, username, password, openai_key):
        self.graph = Neo4jGraph(
            url=neo4j_url,
            username=username,
            password=password
        )
        self.embeddings = OpenAIEmbeddings(api_key=openai_key)
        self.qa_chain = GraphCypherQAChain.from_llm(
            llm=OpenAI(api_key=openai_key),
            graph=self.graph
        )
    
    def query(self, question: str):
        """Query the cybersecurity knowledge graph"""
        return self.qa_chain.run(question)
    
    def add_vector_index(self, node_label: str, property_name: str):
        """Add vector index for semantic search"""
        vector_store = Neo4jVector.from_existing_graph(
            embedding=self.embeddings,
            url=self.graph.url,
            username=self.graph.username,
            password=self.graph.password,
            node_label=node_label,
            text_node_properties=[property_name],
            embedding_node_property="embedding"
        )
        return vector_store

In [ ]:
# Usage example
graph_rag = CybersecurityGraphRAG(
    neo4j_url=os.environ['NEO4J_URI'],
    username=os.environ['NEO4J_USERNAME'],
    password=os.environ['NEO4J_PASSWORD'],
    openai_key=os.environ['OPENAI_API_KEY']
)

# Query examples
result1 = graph_rag.query("What CVEs are related to SQL injection?")
result2 = graph_rag.query("Show me attack patterns that exploit CWE-79")

# Validation and Testing

In [14]:
def validate_graph_structure(driver):
    """Validate graph structure and data quality"""
    with driver.session() as session:
        # Check node counts
        counts = {}
        for label in ['CVE', 'CWE', 'CAPEC', 'Technique']:
            result = session.run(f"MATCH (n:{label}) RETURN count(n) as count")
            counts[label] = result.single()['count']
        
        # Check relationship counts  
        rel_result = session.run("MATCH ()-[r]->() RETURN count(r) as rel_count")
        counts['relationships'] = rel_result.single()['rel_count']
        
        print("Graph Statistics:", counts)
        return counts

def test_graph_queries(driver):
    """Test basic graph queries"""
    with driver.session() as session:
        # Test multi-hop query
        result = session.run("""
            MATCH (cve:CVE)-[:EXPLOITS]->(cwe:CWE)-[:ENABLES]->(capec:CAPEC)
            RETURN cve.id, cwe.id, capec.id
            LIMIT 10
        """)

        print("Multi-hop relationships:")
        for record in result:
            print(f"CVE {record['cve.id']} -> CWE {record['cwe.id']} -> CAPEC {record['capec.id']}")